In [ ]:
import os

# 기준 경로 설정
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"

# "Madinah"라는 단어가 포함된 폴더만 필터링
madinah_folders = [f for f in os.listdir(base_path)
                   if os.path.isdir(os.path.join(base_path, f)) and "Madinah" in f]

# 결과 출력
print("✅ Madinah 관련 폴더 목록:")
for folder in madinah_folders:
    print("-", folder)


In [ ]:
import os

# 기준 경로
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"
madinah_folder = "Madinah_skysatcollect_analytic_sr_udm2-20250403T060043Z-001"
target_path = os.path.join(base_path, madinah_folder)

# 하위 항목 읽기
items = os.listdir(target_path)

print(f"📂 '{madinah_folder}' 폴더 안의 항목:")
for item in items:
    full_path = os.path.join(target_path, item)
    if os.path.isdir(full_path):
        print("  📁", item)  # 폴더
    else:
        print("  📄", item)  # 파일


In [ ]:
import os

# 기본 경로 및 최종 폴더 설정
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"
main_folder = "Madinah_skysatcollect_analytic_sr_udm2-20250403T060043Z-001"
sub_folder = "Madinah_skysatcollect_analytic_sr_udm2"
target_path = os.path.join(base_path, main_folder, sub_folder)

# 하위 항목 출력
items = os.listdir(target_path)

print(f"📂 '{sub_folder}' 폴더 안의 항목:")
for item in items:
    full_path = os.path.join(target_path, item)
    if os.path.isdir(full_path):
        print("  📁", item)
    else:
        print("  📄", item)


In [ ]:
import os

# 기본 경로
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"
folder_1 = "Madinah_skysatcollect_analytic_sr_udm2-20250403T060043Z-001"
folder_2 = "Madinah_skysatcollect_analytic_sr_udm2"
folder_3 = "SkySatCollect"

# 최종 경로 생성
target_path = os.path.join(base_path, folder_1, folder_2, folder_3)

# 해당 경로의 항목 출력
items = os.listdir(target_path)

print(f"📂 '{folder_3}' 폴더 안의 항목:")
for item in items:
    full_path = os.path.join(target_path, item)
    if os.path.isdir(full_path):
        print("  📁", item)
    else:
        print("  📄", item)


In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

# 경로 설정
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"
src_path = os.path.join(base_path, "Madinah_skysatcollect_analytic_sr_udm2-20250403T060043Z-001",
                        "Madinah_skysatcollect_analytic_sr_udm2", "SkySatCollect")
output_dir = os.path.join(base_path, "EVI")
os.makedirs(output_dir, exist_ok=True)

# 대상 날짜 리스트
dates = ["20200213", "20230902"]

# EVI 계산 및 저장
for date in dates:
    # 날짜 포함된 tif 파일 찾기
    target_file = [f for f in os.listdir(src_path) if date in f and f.endswith("_analytic_SR_clip.tif")]
    if not target_file:
        print(f"⚠️ {date} 파일이 없습니다.")
        continue

    tif_path = os.path.join(src_path, target_file[0])
    with rasterio.open(tif_path) as src:
        blue = src.read(1).astype("float32")
        green = src.read(2).astype("float32")
        red = src.read(3).astype("float32")
        nir = src.read(4).astype("float32")
        profile = src.profile
        bounds = src.bounds
        src_transform = src.transform
        src_crs = src.crs

    # EVI 공식 적용
    evi = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0 + 1e-6)
    evi = np.clip(evi, -3, 3)

    # 좌표계 변환
    dst_crs = CRS.from_epsg(4326)
    dst_transform, width, height = calculate_default_transform(
        src_crs, dst_crs, profile["width"], profile["height"], *bounds)

    profile.update({
        "driver": "GTiff",
        "height": height,
        "width": width,
        "transform": dst_transform,
        "crs": dst_crs,
        "count": 1,
        "dtype": "float32"
    })

    out_path = os.path.join(output_dir, f"Madinah_{date}_EVI.tiff")
    with rasterio.open(out_path, "w", **profile) as dst:
        reproject(
            source=evi,
            destination=rasterio.band(dst, 1),
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

    print(f"✅ 저장 완료: {out_path}")


In [ ]:
############### 이 코드 제대로 돌아감 ###############

import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

# 경로 설정
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"
src_path = os.path.join(base_path, "Madinah_skysatcollect_analytic_sr_udm2-20250403T060043Z-001",
                        "Madinah_skysatcollect_analytic_sr_udm2", "SkySatCollect")
output_dir = os.path.join(base_path, "EVI")
os.makedirs(output_dir, exist_ok=True)

# 대상 날짜 리스트
dates = ["20200213", "20230902"]

# EVI 계산 및 저장
for date in dates:
    # 날짜 포함된 tif 파일 찾기
    target_file = [f for f in os.listdir(src_path) if date in f and f.endswith("_analytic_SR_clip.tif")]
    if not target_file:
        print(f"⚠️ {date} 파일이 없습니다.")
        continue

    tif_path = os.path.join(src_path, target_file[0])
    with rasterio.open(tif_path) as src:
        blue = src.read(1).astype("float32") / 10000.0  # 정규화
        green = src.read(2).astype("float32") / 10000.0
        red = src.read(3).astype("float32") / 10000.0
        nir = src.read(4).astype("float32") / 10000.0
        profile = src.profile
        bounds = src.bounds
        src_transform = src.transform
        src_crs = src.crs

    # EVI 공식 적용
    evi = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0 + 1e-6)
    evi = np.clip(evi, -3, 3)

    # 좌표계 변환
    dst_crs = CRS.from_epsg(4326)
    dst_transform, width, height = calculate_default_transform(
        src_crs, dst_crs, profile["width"], profile["height"], *bounds)

    profile.update({
        "driver": "GTiff",
        "height": height,
        "width": width,
        "transform": dst_transform,
        "crs": dst_crs,
        "count": 1,
        "dtype": "float32"
    })

    out_path = os.path.join(output_dir, f"Madinah_{date}_EVI.tif")
    with rasterio.open(out_path, "w", **profile) as dst:
        reproject(
            source=evi,
            destination=rasterio.band(dst, 1),
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

    print(f"✅ 저장 완료: {out_path}")


In [ ]:
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS

# 경로 설정
base_path = r"E:\projects\SpaceUp\Track2\SkySAT"
src_path = os.path.join(base_path, "Madinah_skysatcollect_analytic_sr_udm2-20250403T060043Z-001",
                        "Madinah_skysatcollect_analytic_sr_udm2", "SkySatCollect")
evi_dir = os.path.join(base_path, "EVI")
lai_dir = os.path.join(base_path, "LAI")
os.makedirs(evi_dir, exist_ok=True)
os.makedirs(lai_dir, exist_ok=True)

# 대상 날짜 리스트
dates = ["20200213", "20230902"]

# EVI 및 LAI 계산 및 저장
for date in dates:
    # 날짜 포함된 tif 파일 찾기
    target_file = [f for f in os.listdir(src_path) if date in f and f.endswith("_analytic_SR_clip.tif")]
    if not target_file:
        print(f"⚠️ {date} 파일이 없습니다.")
        continue

    tif_path = os.path.join(src_path, target_file[0])
    with rasterio.open(tif_path) as src:
        blue = src.read(1).astype("float32") / 10000.0  # 정규화
        green = src.read(2).astype("float32") / 10000.0
        red = src.read(3).astype("float32") / 10000.0
        nir = src.read(4).astype("float32") / 10000.0
        profile = src.profile
        bounds = src.bounds
        src_transform = src.transform
        src_crs = src.crs

    # EVI 계산
    evi = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0 + 1e-6)
    evi = np.clip(evi, -3, 3)

    # LAI 계산
    lai = 3.618 * evi - 0.118

    # 좌표계 변환
    dst_crs = CRS.from_epsg(4326)
    dst_transform, width, height = calculate_default_transform(
        src_crs, dst_crs, profile["width"], profile["height"], *bounds)

    profile.update({
        "driver": "GTiff",
        "height": height,
        "width": width,
        "transform": dst_transform,
        "crs": dst_crs,
        "count": 1,
        "dtype": "float32"
    })

    # # EVI 저장
    # evi_path = os.path.join(evi_dir, f"Madinah_{date}_EVI.tif")
    # with rasterio.open(evi_path, "w", **profile) as dst:
    #     reproject(
    #         source=evi,
    #         destination=rasterio.band(dst, 1),
    #         src_transform=src_transform,
    #         src_crs=src_crs,
    #         dst_transform=dst_transform,
    #         dst_crs=dst_crs,
    #         resampling=Resampling.bilinear
    #     )
    # print(f"✅ EVI 저장 완료: {evi_path}")

    # LAI 저장
    lai_path = os.path.join(lai_dir, f"Madinah_{date}_LAI.tif")
    with rasterio.open(lai_path, "w", **profile) as dst:
        reproject(
            source=lai,
            destination=rasterio.band(dst, 1),
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )
    print(f"✅ LAI 저장 완료: {lai_path}")
